# E6 step 1: causal pipeline on data-driven candidate links, then the E6 graph (Kaggle, CPU)

E6 = E4's graph formula (sum |B_k| + P x |B0|) with every discovery step run on **data-driven candidate
links** (each sensor's 25 most correlated training sensors) instead of the road network:

1. `build_candidates.py --k 25`  -> `corr_k25` candidate links (training rows only)
2. `run_var.py --candidates corr_k25 --save-residuals`  -> VAR on those links (lag 3, alpha 1e6, as E1)
3. `run_local_lingam.py`  -> B0 on the same local units (10,000 rows, as E2/E4)
4. `run_local_pem.py`  -> P (same PEM settings as E4: uniform prior 0.5, standardised, L-BFGS-B)
5. `build_graph.py E6`, then a comparison with E4 and the road graph

The E6 A3T-GCN training is step 2, in `ablation-study.ipynb` (`EXPERIMENT = "E6"`).

**Before running**
1. *Add Input* -> the GBA dataset (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add Input* -> the artifacts dataset (`E4__...npz` for the comparison; `sensor_filter_*.json` saves a few minutes).
3. Settings: **Accelerator = None (CPU)**, **Internet = On** (installs `lingam`).
4. **Save Version -> Save & Run All (Commit)**.

**If it stops before the graph is built** (12-hour limit; expected total 7-12 h): the finished steps and
units are saved in `e6_state/` in this version's output. Open a new version, *Add Input* -> *Notebook Output*
of this notebook, and run again: finished steps and units are restored and skipped.

Download from the *Output* tab: `e6_output/` (E6 graph `.npz` + `.json`, summaries, `e6_comparison.json`).

## Settings

In [ ]:
import os

REPO_URL    = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH = "main"

WORKERS            = os.cpu_count()   # Kaggle CPU session: 4
TIME_BUDGET_HOURS  = 11.0             # all steps together; stops before Kaggle's 12-hour limit

## 1. Environment (installs lingam 1.13.0 with the scikit-learn version used for E2/E4)

In [ ]:
import subprocess, sys, time
START = time.time()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lingam==1.13.0", "scikit-learn==1.6.1"], check=True)
r = subprocess.run([sys.executable, "-c",
                    "import lingam, sklearn, numpy, scipy, tensorflow as tf; "
                    "print('lingam', lingam.__version__, '| sklearn', sklearn.__version__, '| numpy', numpy.__version__, "
                    "'| scipy', scipy.__version__, '| tensorflow', tf.__version__)"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0, "package check failed"
print("Python:", sys.version.split()[0], "| CPUs:", os.cpu_count())

## 2. Code and dataset

In [ ]:
import glob, shutil

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
COMMIT = subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

hits = glob.glob("/kaggle/input/**/gba_his_raw_2021.h5", recursive=True)
assert hits, "Add the GBA dataset (Add Input)"
os.makedirs(f"{CODE}/datasets", exist_ok=True)
os.symlink(os.path.dirname(hits[0]), f"{CODE}/datasets/LargeST GBA")

# Optional: sensor-filter cache and the E4 graph (for the comparison) from the artifacts dataset.
GRAPH_DIR = f"{CODE}/outputs/graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(f"{CODE}/outputs/sensor_filter", exist_ok=True)
for path in glob.glob("/kaggle/input/**/*", recursive=True):
    name = os.path.basename(path)
    if not os.path.isfile(path) or "/e6_state/" in path:
        continue
    if name.startswith("sensor_filter_") and name.endswith(".json"):
        shutil.copy(path, f"{CODE}/outputs/sensor_filter/")
    elif name.startswith("E4__") and name.endswith(".npz"):
        shutil.copy(path, f"{GRAPH_DIR}/{name.replace('alpha1e06', 'alpha1e+06')}")
print("sensor filter cache:", os.listdir(f"{CODE}/outputs/sensor_filter"))
print("graphs:", os.listdir(GRAPH_DIR))

## 3. Settings check (E6 must use E4's settings; only the candidate links change)

In [ ]:
sys.path.insert(0, CODE)
from config import config as C

print("Candidates :", C.CANDIDATES_E6)
print("VAR run    :", C.VAR_RUN_E6)
print("LiNGAM run :", C.LINGAM_RUN_E6)
print("PEM run    :", C.PEM_RUN_E6)
print("E6 graph   :", C.FINAL_GRAPHS["E6"])

assert C.CANDIDATES_E6 == "corr_k25"
assert C.VAR_RUN_E6 == C.VAR_RUN + "__cand-corr_k25" and C.VAR_RUN == "lag3_alpha1e+06"
assert C.LINGAM_RUN_E6 == C.VAR_RUN_E6 + "__rows10000"
assert C.PEM_RUN_E6 == C.PEM_RUN.replace(C.VAR_RUN, C.VAR_RUN_E6, 1)          # same PEM settings as E4
assert "eta_0.5" in C.PEM_RUN_E6 and C.PEM_RUN_E6.endswith("_L-BFGS-B_std")
assert (C.PEM_NU0, C.PEM_NU1, C.PEM_TAU, C.PEM_THRESHOLD) == (0.0017, 1.0, 0.01, 0.5)

OUT_ROOT   = f"{CODE}/outputs"
CAND_FILE  = f"{OUT_ROOT}/candidates/{C.CANDIDATES_E6}.npz"
VAR_DIR    = f"{OUT_ROOT}/var_train/{C.VAR_RUN_E6}"
LINGAM_DIR = f"{OUT_ROOT}/lingam/{C.LINGAM_RUN_E6}"
PEM_DIR    = f"{OUT_ROOT}/pem/{C.PEM_RUN_E6}"
GRAPH_FILE = f"{GRAPH_DIR}/{C.FINAL_GRAPHS['E6']}"

# e6_state/<step> <-> outputs folder (state folder names have no "+", which Kaggle drops)
STATE = {"candidates": f"{OUT_ROOT}/candidates", "var": VAR_DIR, "lingam": LINGAM_DIR, "pem": PEM_DIR}
RUN_NAMES = {"var": C.VAR_RUN_E6, "lingam": C.LINGAM_RUN_E6, "pem": C.PEM_RUN_E6}

## 4. Resume: restore finished steps and units from an earlier version's output (if added as input)

In [ ]:
import json
import numpy as np

previous = sorted(glob.glob("/kaggle/input/**/e6_state", recursive=True))
if previous:
    src = previous[0]
    saved = json.load(open(f"{src}/state.json"))
    assert saved["runs"] == RUN_NAMES, f"state is from other settings: {saved['runs']}"
    for step, target in STATE.items():
        if os.path.isdir(f"{src}/{step}"):
            shutil.copytree(f"{src}/{step}", target, dirs_exist_ok=True)
    # A unit file being written when the earlier run was stopped can be broken: delete it so it is redone.
    broken = 0
    for p in glob.glob(f"{LINGAM_DIR}/units/unit_*.npz") + glob.glob(f"{PEM_DIR}/units/unit_*.npz"):
        try:
            with np.load(p) as a:
                a.files
        except Exception:
            os.remove(p); broken += 1
    print("restored from", src, "| written by", saved["commit"], "| broken unit files removed:", broken)
else:
    print("no earlier state: starting from the beginning")

def status():
    return {
        "candidates": os.path.exists(CAND_FILE),
        "var": os.path.exists(f"{VAR_DIR}/metadata.json") and os.path.exists(f"{VAR_DIR}/residuals.npy"),
        "lingam": os.path.exists(f"{LINGAM_DIR}/lingam_result.npz"),
        "lingam_units": len(glob.glob(f"{LINGAM_DIR}/units/unit_*.npz")),
        "pem": os.path.exists(f"{PEM_DIR}/pem_result.npz"),
        "pem_units": len(glob.glob(f"{PEM_DIR}/units/unit_*.npz")),
        "graph": os.path.exists(GRAPH_FILE),
    }
print(status())

## 5. Run the missing steps within the time budget

In [ ]:
DEADLINE = START + TIME_BUDGET_HOURS * 3600
LOG = open(f"{WORK}/e6_pipeline.log", "a")

def run(step, cmd):
    """Run one step; stream its log; kill it at the deadline. Returns True if it finished."""
    if time.time() > DEADLINE:
        print(f"[{step}] skipped: time budget used up"); return False
    print(f"[{step}]", " ".join(cmd), flush=True)
    LOG.write(f"\n===== {step}: {' '.join(cmd)}\n")
    t0, last = time.time(), 0
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        LOG.write(line); LOG.flush()
        low = line.lower()
        if time.time() - last > 300 or "error" in low or "saved" in low or "traceback" in low:
            print(f"  [{step} {(time.time() - t0) / 3600:5.2f} h | total {(time.time() - START) / 3600:5.2f} h] {line.rstrip()}", flush=True)
            last = time.time()
        if time.time() > DEADLINE:
            proc.kill(); proc.wait()
            print(f"[{step}] stopped at the time budget; finished units are kept."); return False
    proc.wait()
    print(f"[{step}] exit code {proc.returncode} after {(time.time() - t0) / 3600:.2f} h", flush=True)
    return proc.returncode == 0

py = [sys.executable, "-u"]
s = status()
ok = True
if ok and not s["candidates"]:
    ok = run("candidates", py + ["build_candidates.py", "--k", "25"])
if ok and not status()["var"]:
    ok = run("var", py + ["run_var.py", "--candidates", C.CANDIDATES_E6, "--save-residuals"])
if ok and not status()["lingam"]:
    ok = run("lingam", py + ["run_local_lingam.py", "--var-run", C.VAR_RUN_E6, "--workers", str(WORKERS)])
if ok and not status()["pem"]:
    ok = run("pem", py + ["run_local_pem.py", "--var-run", C.VAR_RUN_E6, "--workers", str(WORKERS)])
if ok and not status()["graph"]:
    ok = run("graph", py + ["build_graph.py", "E6"])
LOG.close()

s = status()
COMPLETE = s["graph"]
print(s, "| complete:", COMPLETE, f"| hours: {(time.time() - START) / 3600:.2f}")
if os.path.exists(f"{CODE}/outputs/candidates/{C.CANDIDATES_E6}.json"):
    print(open(f"{CODE}/outputs/candidates/{C.CANDIDATES_E6}.json").read()[:600])
print("".join(open(f"{WORK}/e6_pipeline.log").readlines()[-25:]))

## 6. E6 vs E4 and the road graph

In [ ]:
if COMPLETE:
    with np.load(GRAPH_FILE) as a:
        e6, sensors = a["adjacency"].astype(np.float64), a["sensors"]
    road = np.load(f"{CODE}/datasets/LargeST GBA/gba_rn_adj.npy", mmap_mode="r")
    road = np.asarray(road[np.ix_(sensors, sensors)]) > 0
    off = ~np.eye(len(sensors), dtype=bool)

    def share(g):   # median own-signal share after A3T-GCN normalisation (graph stored [target, source])
        a = g.T + np.eye(len(g)); d = a.sum(1)
        return float(np.median(1.0 / d))

    def edges(g):
        return (g != 0) & off

    res = {
        "edges_E6": int(edges(e6).sum()),
        "nodes_without_incoming_E6": int((edges(e6).sum(1) == 0).sum()),
        "share_of_E6_edges_on_road_links": float((edges(e6) & (road | road.T)).sum() / edges(e6).sum()),
        "own_signal_share_E6": share(e6),
    }
    e4_path = f"{GRAPH_DIR}/{C.FINAL_GRAPHS['E4']}"
    if os.path.exists(e4_path):
        with np.load(e4_path) as a:
            e4 = a["adjacency"].astype(np.float64)
            assert np.array_equal(a["sensors"], sensors)
        both = (edges(e4) & edges(e6)).sum()
        res.update({
            "edges_E4": int(edges(e4).sum()),
            "edges_in_both_E4_E6": int(both),
            "jaccard_E4_E6": float(both / (edges(e4) | edges(e6)).sum()),
            "own_signal_share_E4": share(e4),
        })
    for name, d in (("lingam", LINGAM_DIR), ("pem", PEM_DIR)):
        summ = json.load(open(f"{d}/summary.json"))
        res[f"{name}_summary"] = {k: v for k, v in summ.items() if not isinstance(v, (list, dict))}
    print(json.dumps(res, indent=2))
    json.dump(res, open(f"{WORK}/e6_comparison.json", "w"), indent=2)
else:
    print("E6 graph not built yet: run a new version with this version's output added as input (section 4).")

## 7. Collect the outputs to download

In [ ]:
OUT = f"{WORK}/e6_output"
os.makedirs(OUT, exist_ok=True)
files = [f"{WORK}/e6_comparison.json", f"{WORK}/e6_pipeline.log",
         f"{CODE}/outputs/candidates/{C.CANDIDATES_E6}.json", f"{VAR_DIR}/metadata.json",
         f"{LINGAM_DIR}/summary.json", f"{PEM_DIR}/summary.json",
         *glob.glob(f"{GRAPH_DIR}/E6__*")]
for p in files:
    if os.path.exists(p):
        prefix = {VAR_DIR: "var_", LINGAM_DIR: "lingam_", PEM_DIR: "pem_"}.get(os.path.dirname(p), "")
        shutil.copy(p, f"{OUT}/{prefix}{os.path.basename(p)}")
print(sorted(os.listdir(OUT)))
if COMPLETE:
    print("Upload the E6__...npz (and .json) to the artifacts dataset, then run E6 in ablation-study.ipynb.")

## 8. Save state (only if unfinished) and clean up

If the graph was not built, every finished step and unit is copied to `e6_state/` for the next version
(section 4). Then the cloned `implementation` folder is deleted so the Output tab holds only the results.

In [ ]:
if not COMPLETE:
    STATE_DIR = f"{WORK}/e6_state"
    for step, src in STATE.items():
        if os.path.isdir(src):
            shutil.copytree(src, f"{STATE_DIR}/{step}", dirs_exist_ok=True)
    json.dump({"runs": RUN_NAMES, "commit": COMMIT, "status": status()}, open(f"{STATE_DIR}/state.json", "w"), indent=2)
    print("state saved:", status())
shutil.rmtree(CODE)
print("Output contents:", sorted(os.listdir(WORK)))